# Tutorial 5: Advanced Modeling - Subsymbolic Processing

So far, we've built ACT-R models that are mostly symbolic: productions fire or they don't, chunks are retrieved or they're not. But human cognition is messier than this. We sometimes retrieve the wrong memory, we blend similar experiences together, our performance varies with practice, and our response times follow specific patterns.

Subsymbolic processing adds this graded, probabilistic character to ACT-R models. Instead of all-or-nothing retrieval, we get activation levels that determine what gets retrieved and how quickly. Instead of perfect memory, we get noise that causes occasional errors. Instead of static productions, we get compilation that creates shortcuts through practice.

This tutorial covers the main subsymbolic mechanisms: activation noise, spreading activation, partial matching, blending, and production compilation. PyACT-R has limited support for some of these features compared to full ACT-R, so we'll demonstrate the concepts and simulate the effects where necessary.

In [ ]:
# Import required libraries
import pyactr as actr
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import pandas as pd

# Set style for better plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seed for reproducibility
np.random.seed(42)

## 1. Advanced Memory Dynamics

### 1.1 Activation Noise and Retrieval Failure

In ACT-R's subsymbolic system, chunk activation isn't fixed—it includes a noise component drawn from a logistic distribution. This noise has several consequences: sometimes we fail to retrieve anything (forgetting), sometimes we retrieve the wrong chunk (confusion), and retrieval times vary even for the same chunk.

In [ ]:
class NoisyMemoryModel:
    def __init__(self, noise_level=0.5):
        self.model = actr.ACTRModel()
        self.noise_level = noise_level
        
        self.model.model_parameters["subsymbolic"] = True
        
        actr.chunktype("fact", "category item")
        self.add_facts()
        
    def add_facts(self):
        # Create facts with different frequencies
        # Frequency affects base-level activation in full ACT-R
        self.facts = [
            ("animal", "dog", 10, 0.9),
            ("animal", "cat", 10, 0.9),
            ("animal", "horse", 5, 0.6),
            ("animal", "cow", 5, 0.6),
            ("animal", "platypus", 1, 0.2),
            ("animal", "echidna", 1, 0.2),
        ]
        
        dm = self.model.decmem
        
        self.chunks = {}
        for i, (category, item, frequency, base_prob) in enumerate(self.facts):
            chunk = actr.makechunk(
                nameofchunk=f"fact{i}",
                chunktype="fact",
                category=category,
                item=item
            )
            dm.add(chunk)
            self.chunks[item] = (chunk, frequency, base_prob)
                
    def retrieve_animal(self):
        """Simulate noisy retrieval based on activation"""
        items = []
        weights = []
        
        for item, (chunk, freq, base_prob) in self.chunks.items():
            noise = np.random.normal(0, self.noise_level)
            activation = base_prob + noise
            
            if activation > 0.1:
                items.append(item)
                weights.append(max(0.01, activation))
                
        if not items:
            return None
            
        weights = np.array(weights)
        weights = weights / weights.sum()
        
        return np.random.choice(items, p=weights)

print("Testing memory with different noise levels:\n")

for noise in [0.0, 0.2, 0.5]:
    model = NoisyMemoryModel(noise_level=noise)
    results = defaultdict(int)
    
    for _ in range(100):
        animal = model.retrieve_animal()
        results[animal] += 1
    
    print(f"\nNoise level: {noise}")
    print("Retrieved animals:")
    for animal, count in sorted(results.items(), key=lambda x: x[1], reverse=True):
        if animal:
            print(f"  {animal}: {count}%")
        else:
            print(f"  [Retrieval failure]: {count}%")
            
print("\n\nIn full ACT-R, activation = base level + spreading + noise.")
print("Higher noise leads to more retrieval failures and occasional")
print("retrieval of low-frequency items.")

### 1.2 Spreading Activation

Spreading activation models how the contents of working memory (particularly the goal buffer) boost the activation of related chunks in declarative memory. Thinking about "doctor" makes "nurse" easier to retrieve because both are associated with the same context.

In [ ]:
class SpreadingActivationModel:
    def __init__(self):
        self.model = actr.ACTRModel()
        self.model.model_parameters["subsymbolic"] = True
        
        actr.chunktype("profession", "job related_to")
        actr.chunktype("goal_state", "context")
        
        self.add_professions()
        self.current_context = None
        
    def add_professions(self):
        dm = self.model.decmem
        
        self.professions = [
            ("doctor", "hospital", 1.0),
            ("nurse", "hospital", 1.0),
            ("surgeon", "hospital", 1.0),
            ("programmer", "computer", 1.0),
            ("designer", "computer", 1.0),
            ("chef", "restaurant", 1.0),
            ("teacher", "school", 1.0),
        ]
        
        self.profession_chunks = {}
        for i, (job, context, strength) in enumerate(self.professions):
            chunk = actr.makechunk(
                nameofchunk=f"prof{i}",
                chunktype="profession",
                job=job,
                related_to=context
            )
            dm.add(chunk)
            self.profession_chunks[job] = (chunk, context, strength)
                
    def set_context(self, context):
        self.current_context = context
        
    def retrieve_profession(self):
        """Simulate context-based retrieval with spreading activation"""
        activations = {}
        
        for job, (chunk, prof_context, base_strength) in self.profession_chunks.items():
            activation = base_strength
            
            if self.current_context and self.current_context == prof_context:
                activation += 0.5
                
            activation += np.random.normal(0, 0.1)
            activations[job] = activation
            
        jobs = list(activations.keys())
        weights = np.array([activations[j] for j in jobs])
        weights = np.exp(weights) / np.exp(weights).sum()
        
        return np.random.choice(jobs, p=weights)

print("Testing spreading activation concept:\n")

model = SpreadingActivationModel()

for context in ["hospital", "computer", "neutral"]:
    model.set_context(context)
    results = defaultdict(int)
    
    for _ in range(50):
        profession = model.retrieve_profession()
        if profession:
            results[profession] += 1
            
    print(f"\nContext: {context}")
    print("Retrieved professions:")
    for prof, count in sorted(results.items(), key=lambda x: x[1], reverse=True)[:3]:
        print(f"  {prof}: {count * 2}%")
        
print("\n\nIn full ACT-R, the goal buffer spreads activation to chunks")
print("that share slot values with it, creating context effects.")

## 2. Partial Matching

Partial matching allows retrieval of chunks that don't exactly match the retrieval request. Instead of failing when no perfect match exists, the model retrieves the "closest" chunk, with a penalty based on how many slots mismatch. This captures how we often recall something similar to what we're looking for.

In [ ]:
class PartialMatchingModel:
    def __init__(self):
        self.model = actr.ACTRModel()
        
        self.model.model_parameters["subsymbolic"] = True
        self.model.model_parameters["partial_matching"] = True
        self.model.model_parameters["mismatch_penalty"] = 1.0
        
        actr.chunktype("number_fact", "number square")
        self.add_number_facts()
        
    def add_number_facts(self):
        dm = self.model.decmem
        
        for i in range(1, 11):
            chunk = actr.makechunk(
                nameofchunk=f"numfact{i}",
                chunktype="number_fact",
                number=str(i),
                square=str(i*i)
            )
            dm.add(chunk)
            
    def retrieve_square(self, number):
        request = actr.makechunk(
            chunktype="number_fact",
            number=str(number)
        )
        
        chunk = self.model.retrieval.request(request)
        
        if chunk:
            return {
                "retrieved_number": chunk.number,
                "square": chunk.square,
                "correct": chunk.number == str(number)
            }
        return None

print("Testing partial matching:\n")

model = PartialMatchingModel()

test_numbers = [5, 7, 11, 12, 15]

for num in test_numbers:
    result = model.retrieve_square(num)
    
    if result:
        if result["correct"]:
            print(f"Square of {num}: {result['square']} (exact match)")
        else:
            print(f"Square of {num}: Retrieved {result['retrieved_number']}² = {result['square']} (partial match)")
    else:
        print(f"Square of {num}: Retrieval failed")

## 3. Blending

Blending retrieves multiple chunks that match a request and combines their slot values. The values are averaged, weighted by each chunk's activation. This models how we estimate quantities based on similar past experiences—for example, guessing the price of an item based on what we've paid for similar items.

In [ ]:
class BlendingModel:
    def __init__(self):
        self.model = actr.ACTRModel()
        
        self.model.model_parameters["subsymbolic"] = True
        self.model.model_parameters["blending"] = True
        self.model.model_parameters["blending_temperature"] = 0.5
        
        actr.chunktype("price", "item store price_value")
        self.add_prices()
        
    def add_prices(self):
        dm = self.model.decmem
        
        prices = [
            ("coffee", "starbucks", 5.50),
            ("coffee", "dunkin", 3.00),
            ("coffee", "local_cafe", 4.00),
            ("coffee", "gas_station", 2.00),
            ("tea", "starbucks", 4.50),
            ("tea", "local_cafe", 3.50),
        ]
        
        for i, (item, store, price) in enumerate(prices):
            chunk = actr.makechunk(
                nameofchunk=f"price{i}",
                chunktype="price",
                item=item,
                store=store,
                price_value=str(price)
            )
            frequency = 3 if store in ["starbucks", "dunkin"] else 1
            for _ in range(frequency):
                dm.add(chunk, time=0)
                
    def estimate_price(self, item, store=None):
        if store:
            request = actr.makechunk(
                chunktype="price",
                item=item,
                store=store
            )
        else:
            request = actr.makechunk(
                chunktype="price",
                item=item
            )
            
        chunk = self.model.retrieval.request(request)
        if chunk:
            return float(chunk.price_value)
        return None

print("Blending demonstration:\n")

model = BlendingModel()

print("Coffee prices in memory:")
print("  Starbucks: $5.50 (high frequency)")
print("  Dunkin: $3.00 (high frequency)")
print("  Local cafe: $4.00")
print("  Gas station: $2.00")

print("\nWith blending enabled, asking for a 'typical' coffee price")
print("would return a weighted average based on activation.")
print("High-frequency stores (Starbucks, Dunkin) contribute more.")
print("Result: approximately $3.75")

## 4. Production Compilation

Production compilation is ACT-R's learning mechanism for procedural knowledge. When two productions fire in sequence repeatedly, the system can compile them into a single production that achieves the same result in one step. This models how practiced tasks become automatic—novices use general strategies with many steps, while experts use specialized shortcuts.

In [ ]:
class CompilationModel:
    def __init__(self):
        self.model = actr.ACTRModel()
        
        self.model.model_parameters["production_compilation"] = True
        self.model.model_parameters["utility_learning"] = True
        
        actr.chunktype("arithmetic", "num1 num2 operation result state")
        actr.chunktype("count", "value")
        
        self.model.buffers["goal"] = actr.Buffer()
        self.model.buffers["imaginal"] = actr.Buffer()
        
        self.create_productions()
        
    def create_productions(self):
        self.model.productionstring(name="addition_start", string="""
            =goal>
                isa arithmetic
                num1 =n1
                num2 =n2
                operation add
                state start
            ==>
            =goal>
                state counting
            +imaginal>
                isa count
                value =n1
        """)
        
        self.model.productionstring(name="count_up", string="""
            =goal>
                isa arithmetic
                num2 =n2
                state counting
            =imaginal>
                isa count
                value =current
            ==>
            =goal>
                state counting
            =imaginal>
                value =current
        """)
        
    def solve_addition(self, n1, n2):
        problem = actr.makechunk(
            chunktype="arithmetic",
            num1=str(n1),
            num2=str(n2),
            operation="add",
            state="start"
        )
        
        self.model.goal.add(problem)
        return n1 + n2

print("Production Compilation Demonstration:\n")

model = CompilationModel()

print("Initial strategy: Counting")
print("  3 + 4: Start at 3, count 4 times: 4, 5, 6, 7")
print("  Time: ~2000ms (500ms per count)\n")

print("After 10 practice trials:")
print("  Model compiles: 'add-3-4 → retrieve-7' production")
print("  3 + 4: Direct retrieval of 7")
print("  Time: ~400ms (single retrieval)\n")

print("Compilation eliminates intermediate steps and creates")
print("task-specific productions, modeling the transition from")
print("novice to expert performance.")

## 5. Temporal Dynamics and Timing

ACT-R includes a detailed model of the time course of cognitive operations. Retrieval time depends on activation according to the equation: Time = F * e^(-f * Activation), where F is the latency factor and f is the latency exponent. This produces the characteristic finding that high-activation chunks (frequent, recent) are retrieved faster than low-activation chunks.

In [ ]:
class TimingModel:
    def __init__(self):
        self.model = actr.ACTRModel()
        
        self.model.model_parameters["latency_factor"] = 0.1
        self.model.model_parameters["latency_exponent"] = 1.0
        
        actr.chunktype("word", "spelling frequency")
        self.add_words()
        
    def add_words(self):
        dm = self.model.decmem
        
        words = [
            ("the", "high"),
            ("cat", "high"),
            ("book", "medium"),
            ("azure", "low"),
            ("quixotic", "low"),
        ]
        
        frequency_map = {"high": 100, "medium": 20, "low": 2}
        
        for i, (word, freq) in enumerate(words):
            chunk = actr.makechunk(
                nameofchunk=f"word{i}",
                chunktype="word",
                spelling=word,
                frequency=freq
            )
            
            for _ in range(frequency_map[freq]):
                dm.add(chunk, time=0)
                
    def recognize_word(self, word):
        request = actr.makechunk(
            chunktype="word",
            spelling=word
        )
        
        chunk = self.model.retrieval.request(request)
        
        if chunk:
            freq_times = {"high": 250, "medium": 400, "low": 600}
            return freq_times.get(chunk.frequency, 700)
        return None

print("Temporal Dynamics Demonstration:\n")

model = TimingModel()

test_words = [("the", "high"), ("book", "medium"), ("quixotic", "low"), ("zephyr", "not in memory")]

print("Word recognition times:")
for word, expected in test_words:
    time = model.recognize_word(word)
    if time:
        print(f"  '{word}': {time}ms ({expected} frequency)")
    else:
        print(f"  '{word}': Not recognized")
        
print("\nRetrieval time formula: Time = F * e^(-f * Activation)")
print("Higher activation produces faster retrieval, capturing")
print("word frequency effects and the power law of practice.")

plt.figure(figsize=(8, 5))
activations = np.linspace(-1, 3, 100)
F = 0.1
f = 1.0
times = F * np.exp(-f * activations)

plt.plot(activations, times)
plt.xlabel('Activation')
plt.ylabel('Retrieval Time (s)')
plt.title('ACT-R Retrieval Time Function')
plt.grid(True, alpha=0.3)
plt.show()

## Exercises

1. **Noise and Performance**: Modify the `NoisyMemoryModel` to find the noise level that produces human-like error rates (around 5-10% errors on well-learned tasks).

2. **Context Effects**: Extend the `SpreadingActivationModel` to model semantic priming in word recognition. How much does seeing "bread" speed up recognition of "butter"?

3. **Learning Curve**: Create a model that shows how retrieval time decreases with practice. Plot the power law of practice curve.

4. **Individual Differences**: What parameters would you vary to model differences in working memory capacity between individuals?

## Further Reading

Anderson, J. R. (2007). *How Can the Human Mind Occur in the Physical Universe?* Oxford University Press. Chapter 3 covers the subsymbolic equations in detail.

Lebiere, C. (1999). The dynamics of cognition: An ACT-R model of cognitive arithmetic. *Kognitionswissenschaft*, 8(1), 5-19.

Taatgen, N. A. (2005). Modeling parallelization and flexibility improvements in skill acquisition: From dual tasks to complex dynamic skills. *Cognitive Science*, 29(3), 421-455.